In [1]:
# ============================================================
# pilot4_synthetic_disorder_step4_directional_5000
#
# Build clean synthetic Step4T-style implicit-transitive probe samples
# from current Step4T output format.
#
# Target implicit relations and supporting chains are restricted to:
#   left_of, right_of, above, below
#
# Distractor relation sentences may still use all relation labels
# observed in the uploaded reference files.
#
# Output:
#   pilot4_synthetic_disorder_step4_directional_5000.json
#   pilot4_synthetic_disorder_step4_directional_5000.zip
#
# This file can be used as Step5 input.
# ============================================================

import os
import re
import json
import uuid
import random
import shutil
import zipfile
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

from google.colab import files

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Number of implicit-target samples per probed label.
# With 4 target labels, this gives:
#   left_of   2500
#   right_of  2500
#   above     2500
#   below     2500
# Total target samples = 10000.
TARGET_NUM_SAMPLES_PER_TARGET_LABEL = 2500
TARGET_NUM_SAMPLES = TARGET_NUM_SAMPLES_PER_TARGET_LABEL * 4

OUTPUT_BASENAME = "pilot4_synthetic_disorder_step4_lrab_2500"
OUTPUT_JSON = f"{OUTPUT_BASENAME}.json"
OUTPUT_ZIP = f"{OUTPUT_BASENAME}.zip"
STEP5_COMPAT_JSON = f"step4T_{OUTPUT_BASENAME}_implicit_transitive_probe_samples.json"

# Keep work directory aligned with output basename.
WORK_DIR = Path(f"/content/{OUTPUT_BASENAME}")
INPUT_DIR = WORK_DIR / "input_step4"
OUTPUT_DIR = WORK_DIR / "output"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ACCEPTED_INPUT_SUFFIXES = {".json", ".jsonl", ".zip"}

# Scene split is designed to match your current Step6 setup:
#   train scenes = FloorPlan1-4
#   test scenes  = FloorPlan5-6
TRAIN_SCENES = ["FloorPlan1", "FloorPlan2", "FloorPlan3", "FloorPlan4"]
TEST_SCENES = ["FloorPlan5", "FloorPlan6"]
ALL_SCENES = TRAIN_SCENES + TEST_SCENES

# Use disjoint object pools for train and test scenes.
TRAIN_POOL_NAME = "train"
TEST_POOL_NAME = "test"

# ------------------------------------------------------------
# Clean implicit target labels
# ------------------------------------------------------------

# Only these labels can be used as implicit target relations.
# They are also the only labels allowed in supporting relation chains.
IMPLICIT_TARGET_LABELS = {
    "left_of",
    "right_of",
    "above",
    "below",
}

# Fixed target-label order for reproducibility.
LABELS = [
    "left_of",
    "right_of",
    "above",
    "below",
]

# These are filled after loading reference files in Cell 2.
# They are used only for distractor generation.
ALL_REFERENCE_LABELS = []
DISTRACTOR_RELATION_LABELS = []

IMPLICIT_RULES = [
    "chain_same_direction",
    "shared_anchor_opposite_sides",
    "anchor_between_reversed_surface_form",
]

# Include all inverse pairs because distractors may use all reference labels.
INVERSE_RELATION_MAP = {
    "left_of": "right_of",
    "right_of": "left_of",

    "above": "below",
    "below": "above",

    "in": "contains",
    "contains": "in",

    "on": "supports",
    "supports": "on",

    # near is symmetric.
    "near": "near",
}

print("Clean directional synthetic Step4T construction config loaded.")
print("TARGET_NUM_SAMPLES_PER_TARGET_LABEL:", TARGET_NUM_SAMPLES_PER_TARGET_LABEL)
print("TARGET_NUM_SAMPLES:", TARGET_NUM_SAMPLES)
print("Implicit target labels:", sorted(IMPLICIT_TARGET_LABELS))
print("Output:", OUTPUT_JSON)
print("Work dir:", WORK_DIR)

Clean directional synthetic Step4T construction config loaded.
TARGET_NUM_SAMPLES_PER_TARGET_LABEL: 2500
TARGET_NUM_SAMPLES: 10000
Implicit target labels: ['above', 'below', 'left_of', 'right_of']
Output: pilot4_synthetic_disorder_step4_lrab_25000.json
Work dir: /content/pilot4_synthetic_disorder_step4_lrab_25000


In [2]:
# ============================================================
# Upload current Step4T output
#
# Accepted:
#   - .zip containing step4T_*_implicit_transitive_probe_samples.json
#   - one or more .json files
#   - .jsonl files
# ============================================================

print("Upload the six FloorPlan Step4 probe files and the old synthetic file, or a zip containing them.")
uploaded = files.upload()

if not uploaded:
    raise FileNotFoundError("No files uploaded.")

# Clear old input files.
for p in INPUT_DIR.glob("*"):
    if p.is_file():
        p.unlink()
    elif p.is_dir():
        shutil.rmtree(p)

for name in uploaded.keys():
    src = Path("/content") / name
    dst = INPUT_DIR / name
    shutil.copy(str(src), str(dst))

print("\nUploaded files copied to:")
for p in sorted(INPUT_DIR.glob("*")):
    print("-", p)

Upload the six FloorPlan Step4 probe files and the old synthetic file, or a zip containing them.


Saving 新建文件夹 (3).zip to 新建文件夹 (3).zip

Uploaded files copied to:
- /content/pilot4_synthetic_disorder_step4_lrab_25000/input_step4/新建文件夹 (3).zip


In [3]:
# ============================================================
# Load current Step4T output samples
# ============================================================

def load_json_or_jsonl(path: Path) -> List[Dict[str, Any]]:
    if path.suffix == ".jsonl":
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows

    if path.suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, list):
            return data

        for key in ["records", "samples", "data"]:
            if isinstance(data, dict) and key in data and isinstance(data[key], list):
                return data[key]

        raise ValueError(f"Unsupported JSON structure in {path}")

    raise ValueError(f"Unsupported file type: {path}")


def extract_all_input_files(input_dir: Path) -> List[Path]:
    extracted_dir = input_dir / "_extracted"
    extracted_dir.mkdir(exist_ok=True)

    for zip_path in input_dir.glob("*.zip"):
        with zipfile.ZipFile(str(zip_path), "r") as zf:
            zf.extractall(str(extracted_dir))

    files_found = []
    files_found.extend(input_dir.rglob("*.json"))
    files_found.extend(input_dir.rglob("*.jsonl"))

    # Exclude files inside output if rerun.
    files_found = [
        p for p in files_found
        if OUTPUT_DIR not in p.parents
    ]

    return sorted(set(files_found))


input_files = extract_all_input_files(INPUT_DIR)

if not input_files:
    raise FileNotFoundError("No .json or .jsonl files found in uploaded Step4T output.")

print("Input files found:")
for p in input_files:
    print("-", p.relative_to(INPUT_DIR))

seed_rows = []

for p in input_files:
    try:
        rows = load_json_or_jsonl(p)
        seed_rows.extend(rows)
        print(f"Loaded {len(rows)} rows from {p.name}")
    except Exception as e:
        print(f"Skipped {p}: {e}")

if not seed_rows:
    raise ValueError("No Step4T rows could be loaded.")

print("\nTotal rows loaded:", len(seed_rows))

# ============================================================
# Split real reference rows from existing synthetic rows
# ============================================================

def is_synthetic_row(row: Dict[str, Any]) -> bool:
    meta = row.get("synthetic_metadata", {}) or {}
    return bool(meta.get("is_synthetic")) or row.get("generation_mode") == "synthetic"


reference_rows = [r for r in seed_rows if not is_synthetic_row(r)]
current_synthetic_reference_rows = [r for r in seed_rows if is_synthetic_row(r)]

if not reference_rows:
    reference_rows = seed_rows

print("Reference real rows used:", len(reference_rows))
print("Current synthetic reference rows used for minimum label counts:", len(current_synthetic_reference_rows))


# ============================================================
# Extract object type, uid, label, and text-length references
# ============================================================

def collect_object_types_and_uids(rows: List[Dict[str, Any]]) -> Tuple[List[str], List[str]]:
    object_types = set()
    object_uids = set()

    for row in rows:
        pp = row.get("probe_pair", {})
        ev = row.get("evidence", {})

        for key in [
            "probe_subject_type",
            "probe_object_type",
        ]:
            if pp.get(key):
                object_types.add(str(pp[key]))

        for key in [
            "probe_subject_uid",
            "probe_object_uid",
            "probe_subject_mention_text",
            "probe_object_mention_text",
        ]:
            if pp.get(key):
                object_uids.add(str(pp[key]))

        for sr in ev.get("supporting_relations", []) or []:
            for key in ["subject_type", "object_type"]:
                if sr.get(key):
                    object_types.add(str(sr[key]))

            for key in ["subject_uid", "object_uid"]:
                if sr.get(key):
                    object_uids.add(str(sr[key]))

    return sorted(object_types), sorted(object_uids)


seed_types, seed_uids = collect_object_types_and_uids(reference_rows)

fallback_types = [
    "Apple", "Mug", "Plate", "Bowl", "Cup", "Fork", "Spoon", "Knife",
    "Book", "Bottle", "Box", "Pan", "Pot", "Kettle", "Lettuce", "Tomato",
    "Potato", "SoapBottle", "WineBottle", "PepperShaker", "SaltShaker",
    "Spatula", "Cabinet", "CounterTop", "DishSponge", "CoffeeMachine",
]

if len(seed_types) < 10:
    object_types = fallback_types
else:
    object_types = seed_types

print("\nObject types used for synthetic generation:")
print(object_types)

print("\nNumber of seed object uids:", len(seed_uids))


# ============================================================
# Label statistics
# ============================================================

reference_label_counts = Counter(
    str(r.get("probe_pair", {}).get("probe_relation_label"))
    for r in reference_rows
    if r.get("probe_pair", {}).get("probe_relation_label")
)

current_synthetic_label_counts = Counter(
    str(r.get("probe_pair", {}).get("probe_relation_label"))
    for r in current_synthetic_reference_rows
    if r.get("probe_pair", {}).get("probe_relation_label")
)


def count_sentence_like_units(text: str) -> int:
    text = (text or "").strip()
    if not text:
        return 0
    return len([s for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()])


REFERENCE_SENTENCE_COUNTS = [
    count_sentence_like_units(r.get("text", ""))
    for r in reference_rows
    if r.get("text")
]

# ============================================================
# Separate implicit target labels from distractor labels
# ============================================================

ALL_REFERENCE_LABELS = sorted(reference_label_counts.keys())

if not ALL_REFERENCE_LABELS:
    raise ValueError(
        "No relation labels found in uploaded reference rows. "
        "Cannot build distractor relation label set."
    )

missing_target_labels = sorted(
    label for label in IMPLICIT_TARGET_LABELS
    if label not in reference_label_counts
)

if missing_target_labels:
    raise ValueError(
        "The uploaded reference files do not contain all required "
        "directional implicit target labels.\n"
        f"Required target labels: {sorted(IMPLICIT_TARGET_LABELS)}\n"
        f"Missing target labels: {missing_target_labels}\n"
        f"Available reference labels: {ALL_REFERENCE_LABELS}"
    )

# Target labels are restricted to clean directional relations.
LABELS = [
    "left_of",
    "right_of",
    "above",
    "below",
]

# Distractor labels retain all labels found in reference files.
# Template availability is checked in Cell 5.
DISTRACTOR_RELATION_LABELS = list(ALL_REFERENCE_LABELS)

filtered_reference_label_counts = Counter({
    label: reference_label_counts[label]
    for label in LABELS
    if label in reference_label_counts
})

filtered_current_synthetic_label_counts = Counter({
    label: current_synthetic_label_counts[label]
    for label in LABELS
    if label in current_synthetic_label_counts
})

print("\nAll reference labels:", ALL_REFERENCE_LABELS)
print("All reference label counts:", dict(reference_label_counts))

print("\nImplicit target labels used:", LABELS)
print("Implicit target reference counts:", dict(filtered_reference_label_counts))

print("\nDistractor relation labels before template filtering:")
print(DISTRACTOR_RELATION_LABELS)

print("\nAll current synthetic min label counts:", dict(current_synthetic_label_counts))
print("Directional current synthetic min label counts used:", dict(filtered_current_synthetic_label_counts))

print("\nReference sentence-count distribution:", dict(Counter(REFERENCE_SENTENCE_COUNTS)))

Input files found:
- _extracted/FloorPlan1_step4_probe_samples.json
- _extracted/FloorPlan2_step4_probe_samples.json
- _extracted/FloorPlan3_step4_probe_samples.json
- _extracted/FloorPlan4_step4_probe_samples.json
- _extracted/FloorPlan5_step4_probe_samples.json
- _extracted/FloorPlan6_step4_probe_samples.json
- _extracted/pilot4_synthetic_step4_lr_5000.json
Loaded 708 rows from FloorPlan1_step4_probe_samples.json
Loaded 582 rows from FloorPlan2_step4_probe_samples.json
Loaded 564 rows from FloorPlan3_step4_probe_samples.json
Loaded 580 rows from FloorPlan4_step4_probe_samples.json
Loaded 708 rows from FloorPlan5_step4_probe_samples.json
Loaded 640 rows from FloorPlan6_step4_probe_samples.json
Loaded 5000 rows from pilot4_synthetic_step4_lr_5000.json

Total rows loaded: 8782
Reference real rows used: 3782
Current synthetic reference rows used for minimum label counts: 5000

Object types used for synthetic generation:
['Apple', 'Book', 'Bottle', 'Bowl', 'Bread', 'ButterKnife', 'Cabinet

In [4]:
# ============================================================
# Surface templates for spatial relations
#
# Important:
#   LABELS is read from the real Step4 reference files in Cell 2.
#
# This cell defines:
#   1. RELATION_TEMPLATES:
#        direct surface templates for real Step4 labels
#   2. INVERSE_RELATION_MAP:
#        inverse relation pairs used for inverse surface realization
#   3. DIRECT_SURFACE_TEMPLATES:
#        direct templates used by realize_relation_sentence()
#   4. INVERSE_SURFACE_TEMPLATES:
#        automatically derived inverse templates
#   5. valid_implicit_rules_for_label():
#        rule selector used by Cell 6
#
# Expected labels from the six FloorPlan Step4 files:
#   above, below, contains, in, left_of, near, on, right_of, supports
# ============================================================

RELATION_TEMPLATES = {
    "in": [
        "{subj} is in {obj}.",
        "{subj} is inside {obj}.",
        "{subj} is contained in {obj}.",
        "{subj} is kept in {obj}.",
        "{subj} sits inside {obj}.",
        "{subj} can be found in {obj}.",
        "{subj} is placed inside {obj}.",
        "You can find {subj} in {obj}.",
        "Inside {obj}, there is {subj}.",
        "Inside {obj}, you can see {subj}.",
    ],
    "on": [
        "{subj} is on {obj}.",
        "{subj} is on top of {obj}.",
        "{subj} is resting on {obj}.",
        "{subj} sits on {obj}.",
        "{subj} has been placed on {obj}.",
        "{subj} lies on {obj}.",
        "You can see {subj} on {obj}.",
        "On {obj}, there is {subj}.",
        "On top of {obj}, there is {subj}.",
    ],
    "left_of": [
        "{subj} is to the left of {obj}.",
        "{subj} is on the left side of {obj}.",
        "{subj} sits to the left of {obj}.",
        "{subj} is positioned to the left of {obj}.",
        "{subj} can be seen to the left of {obj}.",
        "To the left of {obj}, there is {subj}.",
    ],
    "right_of": [
        "{subj} is to the right of {obj}.",
        "{subj} is on the right side of {obj}.",
        "{subj} sits to the right of {obj}.",
        "{subj} is positioned to the right of {obj}.",
        "{subj} can be seen to the right of {obj}.",
        "To the right of {obj}, there is {subj}.",
    ],
    "above": [
        "{subj} is above {obj}.",
        "{subj} is positioned above {obj}.",
        "{subj} is located above {obj}.",
        "{subj} sits above {obj}.",
        "{subj} appears above {obj}.",
        "Above {obj}, there is {subj}.",
        "Directly above {obj}, you can see {subj}.",
    ],
    "below": [
        "{subj} is below {obj}.",
        "{subj} is underneath {obj}.",
        "{subj} is located below {obj}.",
        "{subj} sits below {obj}.",
        "{subj} appears below {obj}.",
        "Below {obj}, there is {subj}.",
        "Directly below {obj}, you can see {subj}.",
    ],
    "near": [
        "{subj} is near {obj}.",
        "{subj} is close to {obj}.",
        "{subj} is nearby {obj}.",
        "{subj} is not far from {obj}.",
        "{subj} is positioned near {obj}.",
        "{subj} is located close to {obj}.",
        "{subj} can be found near {obj}.",
        "Near {obj}, there is {subj}.",
    ],
    "supports": [
        "{subj} supports {obj}.",
        "{subj} is holding up {obj}.",
        "{subj} has {obj} on top of it.",
        "{obj} is resting on {subj}.",
        "On {subj}, there is {obj}.",
    ],
    "contains": [
        "{subj} contains {obj}.",
        "{subj} has {obj} inside it.",
        "{subj} holds {obj}.",
        "{subj} includes {obj}.",
        "Inside {subj}, there is {obj}.",
        "{obj} is inside {subj}.",
    ],
}


def relation_label_to_phrase(label: str) -> str:
    """
    Convert a relation label into a readable phrase.
    Used only for defensive fallback templates.
    """
    explicit_phrases = {
        "left_of": "to the left of",
        "right_of": "to the right of",
        "above": "above",
        "below": "below",
        "in": "inside",
        "inside": "inside",
        "contains": "contains",
        "on": "on",
        "supports": "supports",
        "near": "near",
    }

    if label in explicit_phrases:
        return explicit_phrases[label]

    return label.replace("_", " ")


def infer_inverse_relation(label: str) -> str:
    """
    Infer inverse relation for labels covered by the FloorPlan Step4 files.

    Relation pairs:
      left_of  <-> right_of
      above    <-> below
      in       <-> contains
      on       <-> supports
      near     <-> near
    """
    inverse_map = {
        "left_of": "right_of",
        "right_of": "left_of",

        "above": "below",
        "below": "above",

        "in": "contains",
        "inside": "contains",
        "contains": "in",

        "on": "supports",
        "supports": "on",

        "near": "near",
    }

    return inverse_map.get(label, label)


# ============================================================
# Build surface template dictionaries
#
# Important:
#   LABELS now means implicit target labels only:
#       left_of, right_of, above, below
#
#   DISTRACTOR_RELATION_LABELS contains all reference labels and is
#   used only for distractor relation sentences.
# ============================================================

# Reset inverse map using all labels that may appear either as targets
# or as distractors.
all_relation_labels_for_templates = set(LABELS) | set(DISTRACTOR_RELATION_LABELS)

for label in list(all_relation_labels_for_templates):
    all_relation_labels_for_templates.add(infer_inverse_relation(label))

INVERSE_RELATION_MAP.clear()

for label in sorted(all_relation_labels_for_templates):
    INVERSE_RELATION_MAP[label] = infer_inverse_relation(label)

# Ensure inverse labels also map back.
for label, inv in list(INVERSE_RELATION_MAP.items()):
    if inv not in INVERSE_RELATION_MAP:
        INVERSE_RELATION_MAP[inv] = label


def build_fallback_templates_for_relation(label: str) -> List[str]:
    """
    Fallback direct templates for labels not explicitly listed in RELATION_TEMPLATES.
    """
    phrase = relation_label_to_phrase(label)

    return [
        f"{{subj}} is {phrase} {{obj}}.",
        f"{{subj}} appears {phrase} {{obj}}.",
        f"{{subj}} can be seen {phrase} {{obj}}.",
    ]


def swap_template_arguments(template: str) -> str:
    """
    Convert a direct template for inverse(label) into an inverse surface
    template for label.
    """
    return (
        template
        .replace("{subj}", "__TMP_SUBJ__")
        .replace("{obj}", "__TMP_OBJ__")
        .replace("__TMP_SUBJ__", "{obj}")
        .replace("__TMP_OBJ__", "{subj}")
    )


def get_direct_templates(label: str) -> List[str]:
    """
    Return direct templates for a label.
    """
    if label in RELATION_TEMPLATES:
        return RELATION_TEMPLATES[label]

    return build_fallback_templates_for_relation(label)


def get_inverse_templates(label: str) -> List[str]:
    """
    Return inverse surface templates for a logical relation label.

    Logical relation:
      subj label obj

    Inverse surface text:
      obj inverse(label) subj
    """
    inverse_label = INVERSE_RELATION_MAP.get(label, label)
    inverse_direct_templates = get_direct_templates(inverse_label)

    return [
        swap_template_arguments(template)
        for template in inverse_direct_templates
    ]


# Build template dictionaries for:
#   1. clean target labels
#   2. all reference distractor labels
#   3. their inverse labels
all_template_labels = set(LABELS) | set(DISTRACTOR_RELATION_LABELS)

for label in list(all_template_labels):
    all_template_labels.add(INVERSE_RELATION_MAP.get(label, label))

DIRECT_SURFACE_TEMPLATES = {
    label: get_direct_templates(label)
    for label in sorted(all_template_labels)
}

INVERSE_SURFACE_TEMPLATES = {
    label: get_inverse_templates(label)
    for label in sorted(all_template_labels)
}


INTRO_TEMPLATES = [
    "In this local scene,",
    "In this part of the room,",
    "Within the described area,",
    "In the observed region,",
    "Looking at this local arrangement,",
    "In this synthetic spatial description,",
]


def valid_implicit_rules_for_label(label: str) -> List[str]:
    """
    Return valid implicit construction rules for each target label.

    Only clean directional labels can become implicit targets and
    generate supporting relation chains:

        left_of, right_of, above, below

    Other labels may appear as distractor relation sentences, but they
    are not used as implicit target labels.
    """
    if label in IMPLICIT_TARGET_LABELS:
        return IMPLICIT_RULES

    return []


print("Template library loaded.")
print("Implicit target labels:", LABELS)
print("Distractor relation labels:", DISTRACTOR_RELATION_LABELS)

print("\nInverse relation map used:")
for label in sorted(all_template_labels):
    print(f"  {label} -> {INVERSE_RELATION_MAP.get(label, label)}")

print("\nTemplate labels:")
print(sorted(all_template_labels))

print("\nDirect template counts:")
for label in sorted(DIRECT_SURFACE_TEMPLATES.keys()):
    print(f"  {label}: {len(DIRECT_SURFACE_TEMPLATES[label])}")

print("\nInverse template counts:")
for label in sorted(INVERSE_SURFACE_TEMPLATES.keys()):
    print(f"  {label}: {len(INVERSE_SURFACE_TEMPLATES[label])}")

Template library loaded.
Implicit target labels: ['left_of', 'right_of', 'above', 'below']
Distractor relation labels: ['above', 'below', 'contains', 'in', 'left_of', 'near', 'on', 'right_of', 'supports']

Inverse relation map used:
  above -> below
  below -> above
  contains -> in
  in -> contains
  left_of -> right_of
  near -> near
  on -> supports
  right_of -> left_of
  supports -> on

Template labels:
['above', 'below', 'contains', 'in', 'left_of', 'near', 'on', 'right_of', 'supports']

Direct template counts:
  above: 7
  below: 7
  contains: 6
  in: 10
  left_of: 6
  near: 8
  on: 9
  right_of: 6
  supports: 5

Inverse template counts:
  above: 7
  below: 7
  contains: 10
  in: 6
  left_of: 6
  near: 8
  on: 5
  right_of: 6
  supports: 9


In [5]:
# ============================================================
# Build disjoint train/test object pools
# ============================================================

def normalize_type_to_alias_prefix(type_name: str) -> str:
    # CamelCase / mixed type -> lower snake-ish prefix
    s = re.sub(r"[^A-Za-z0-9]+", "_", type_name).strip("_")
    s = re.sub(r"(?<!^)(?=[A-Z])", "_", s).lower()
    s = re.sub(r"_+", "_", s)
    return s or "object"


def build_object_pool(
    pool_name: str,
    object_types: List[str],
    num_objects: int,
) -> List[Dict[str, str]]:
    objects = []

    if pool_name == TRAIN_POOL_NAME:
        pool_offset = 0
    elif pool_name == TEST_POOL_NAME:
        pool_offset = 100000
    else:
        raise ValueError(f"Unknown pool_name: {pool_name}")

    for i in range(num_objects):
        type_name = object_types[i % len(object_types)]
        prefix = normalize_type_to_alias_prefix(type_name)

        # No train_/test_ prefix in uid or mention.
        # This uid is what appears in text.
        visible_index = pool_offset + i
        uid = f"{prefix}_{visible_index:06d}"

        # Keep split marker only in metadata/id, not in text.
        obj_id = f"Synthetic{type_name}|{pool_name}|{i:04d}"

        objects.append({
            "uid": uid,
            "mention": uid,
            "type": type_name,
            "id": obj_id,
            "pool": pool_name,
        })

    random.shuffle(objects)
    return objects


# Need enough objects so samples do not repeatedly use the same small object set.
# 1800 per train pool and 900 per test pool are enough for 5000 samples.
TRAIN_OBJECTS = build_object_pool(TRAIN_POOL_NAME, object_types, num_objects=1800)
TEST_OBJECTS = build_object_pool(TEST_POOL_NAME, object_types, num_objects=900)

TRAIN_UIDS = {o["uid"] for o in TRAIN_OBJECTS}
TEST_UIDS = {o["uid"] for o in TEST_OBJECTS}

assert TRAIN_UIDS.isdisjoint(TEST_UIDS)

print("Train object pool size:", len(TRAIN_OBJECTS))
print("Test object pool size:", len(TEST_OBJECTS))
print("Train/test object uid overlap:", len(TRAIN_UIDS & TEST_UIDS))
print("\nExample train objects:", TRAIN_OBJECTS[:5])
print("\nExample test objects:", TEST_OBJECTS[:5])

Train object pool size: 1800
Test object pool size: 900
Train/test object uid overlap: 0

Example train objects: [{'uid': 'fridge_001475', 'mention': 'fridge_001475', 'type': 'Fridge', 'id': 'SyntheticFridge|train|1475', 'pool': 'train'}, {'uid': 'mug_000288', 'mention': 'mug_000288', 'type': 'Mug', 'id': 'SyntheticMug|train|0288', 'pool': 'train'}, {'uid': 'pan_001225', 'mention': 'pan_001225', 'type': 'Pan', 'id': 'SyntheticPan|train|1225', 'pool': 'train'}, {'uid': 'vase_001193', 'mention': 'vase_001193', 'type': 'Vase', 'id': 'SyntheticVase|train|1193', 'pool': 'train'}, {'uid': 'drawer_000639', 'mention': 'drawer_000639', 'type': 'Drawer', 'id': 'SyntheticDrawer|train|0639', 'pool': 'train'}]

Example test objects: [{'uid': 'fork_100018', 'mention': 'fork_100018', 'type': 'Fork', 'id': 'SyntheticFork|test|0018', 'pool': 'test'}, {'uid': 'shelving_unit_100193', 'mention': 'shelving_unit_100193', 'type': 'ShelvingUnit', 'id': 'SyntheticShelvingUnit|test|0193', 'pool': 'test'}, {'uid

In [6]:
# ============================================================
# Core synthetic sample generation
# ============================================================

def find_all_char_spans(text: str, mention: str) -> List[Dict[str, int]]:
    spans = []
    start = 0

    while True:
        idx = text.find(mention, start)
        if idx == -1:
            break

        spans.append({
            "start": idx,
            "end": idx + len(mention),
        })
        start = idx + len(mention)

    return spans


def choose_three_distinct_objects(object_pool: List[Dict[str, str]]) -> Tuple[Dict[str, str], Dict[str, str], Dict[str, str]]:
    if len(object_pool) < 3:
        raise ValueError("Object pool must contain at least three objects.")

    return tuple(random.sample(object_pool, 3))


def inverse_relation(relation: str) -> str:
    if relation not in INVERSE_RELATION_MAP:
        INVERSE_RELATION_MAP[relation] = infer_inverse_relation(relation)

    return INVERSE_RELATION_MAP[relation]


def ensure_templates_for_relation(relation: str) -> None:
    """
    Defensive helper. If a relation appears during generation but was
    not precomputed, create templates on demand.
    """
    if relation not in DIRECT_SURFACE_TEMPLATES:
        DIRECT_SURFACE_TEMPLATES[relation] = get_direct_templates(relation)

    if relation not in INVERSE_SURFACE_TEMPLATES:
        INVERSE_SURFACE_TEMPLATES[relation] = get_inverse_templates(relation)


def realize_relation_sentence(
    subj: str,
    relation: str,
    obj: str,
    force_surface_mode: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Returns a sentence plus metadata.

    Logical relation remains:
        subj relation obj

    Surface form may be:
        direct: subj relation obj
        inverse: obj inverse(relation) subj
    """

    ensure_templates_for_relation(relation)

    if force_surface_mode is None:
        surface_mode = random.choice(["direct", "inverse"])
    else:
        surface_mode = force_surface_mode

    if surface_mode == "direct":
        templates = DIRECT_SURFACE_TEMPLATES[relation]
        template = random.choice(templates)
        sentence = template.format(subj=subj, obj=obj)

        return {
            "sentence": sentence,
            "template": template,
            "surface_mode": "direct",
            "was_swapped_for_text": False,
            "surface_subject_uid": subj,
            "surface_relation": relation,
            "surface_object_uid": obj,
        }

    if surface_mode == "inverse":
        inv = inverse_relation(relation)
        templates = INVERSE_SURFACE_TEMPLATES[relation]
        template = random.choice(templates)
        sentence = template.format(subj=subj, obj=obj)

        return {
            "sentence": sentence,
            "template": template,
            "surface_mode": "inverse",
            "was_swapped_for_text": True,
            "surface_subject_uid": obj,
            "surface_relation": inv,
            "surface_object_uid": subj,
        }

    raise ValueError(f"Unknown surface mode: {surface_mode}")


def build_support_edges(
    A: Dict[str, str],
    B: Dict[str, str],
    C: Dict[str, str],
    target_relation: str,
    implicit_rule: str,
) -> List[Tuple[Dict[str, str], str, Dict[str, str]]]:
    """
    Construct two explicit support edges that imply:

        A target_relation C

    In the clean directional setting, target_relation must be one of:
        left_of, right_of, above, below
    """

    if target_relation not in IMPLICIT_TARGET_LABELS:
        raise ValueError(
            f"Invalid implicit target relation: {target_relation}. "
            f"Allowed: {sorted(IMPLICIT_TARGET_LABELS)}"
        )

    inv = inverse_relation(target_relation)

    if implicit_rule == "chain_same_direction":
        return [
            (A, target_relation, B),
            (B, target_relation, C),
        ]

    if implicit_rule == "shared_anchor_opposite_sides":
        return [
            (A, target_relation, B),
            (C, inv, B),
        ]

    if implicit_rule == "anchor_between_reversed_surface_form":
        return [
            (B, inv, A),
            (B, target_relation, C),
        ]

    raise ValueError(f"Unknown implicit_rule: {implicit_rule}")


def make_supporting_relation_record(
    subj: Dict[str, str],
    relation: str,
    obj: Dict[str, str],
    sentence_info: Dict[str, Any],
    generation_mode: str = "synthetic",
    template_index: int = -1,
) -> Dict[str, Any]:
    return {
        "sentence": sentence_info["sentence"],
        "template": sentence_info["template"],
        "template_index": template_index,
        "generation_mode": generation_mode,

        "subject_uid": subj["uid"],
        "subject_id": subj["id"],
        "subject_type": subj["type"],
        "relation": relation,
        "object_uid": obj["uid"],
        "object_id": obj["id"],
        "object_type": obj["type"],

        "was_swapped_for_text": sentence_info["was_swapped_for_text"],
        "surface_mode": sentence_info["surface_mode"],
        "surface_subject_uid": sentence_info["surface_subject_uid"],
        "surface_relation": sentence_info["surface_relation"],
        "surface_object_uid": sentence_info["surface_object_uid"],

        "source_relation": {
            "subject_id": subj["id"],
            "relation": relation,
            "object_id": obj["id"],
            "relation_family": "synthetic_geometric",
        },
    }


def get_available_distractor_relation_labels() -> List[str]:
    """
    Return all reference labels that can be realized by RELATION_TEMPLATES.

    These labels are used only for distractor sentences.
    They do not define target labels and do not form support chains.
    """
    labels = [
        label for label in DISTRACTOR_RELATION_LABELS
        if label in DIRECT_SURFACE_TEMPLATES
    ]

    if not labels:
        raise ValueError(
            "No distractor relation labels have templates.\n"
            f"DISTRACTOR_RELATION_LABELS={DISTRACTOR_RELATION_LABELS}\n"
            f"Available template labels={sorted(DIRECT_SURFACE_TEMPLATES.keys())}"
        )

    return labels

# ============================================================
# Balanced distractor relation label sampler
#
# Goal:
#   Use all reference relation labels as distractor labels,
#   while keeping their logical-label distribution approximately balanced.
#
# This controls only:
#   evidence["distractor_relation_labels"]
#
# It does NOT affect:
#   probe_pair["probe_relation_label"]
#   evidence["supporting_relations"][...]["relation"]
# ============================================================

DISTRACTOR_LABEL_CYCLE = []
DISTRACTOR_LABEL_CURSOR = 0


def reset_balanced_distractor_label_sampler() -> None:
    """
    Initialize a shuffled balanced cycle over all available distractor labels.

    Each cycle contains every available distractor label exactly once.
    Repeating this cycle keeps the global distractor logical-label
    distribution approximately balanced.
    """
    global DISTRACTOR_LABEL_CYCLE
    global DISTRACTOR_LABEL_CURSOR

    DISTRACTOR_LABEL_CYCLE = get_available_distractor_relation_labels()

    if not DISTRACTOR_LABEL_CYCLE:
        raise ValueError("Cannot initialize distractor sampler: no available labels.")

    random.shuffle(DISTRACTOR_LABEL_CYCLE)
    DISTRACTOR_LABEL_CURSOR = 0

    print("Balanced distractor label sampler initialized.")
    print("Distractor label cycle:", DISTRACTOR_LABEL_CYCLE)


def sample_balanced_distractor_relation_label() -> str:
    """
    Sample one distractor logical relation label from a balanced cycle.

    When the cycle is exhausted, it is reshuffled and reused.
    """
    global DISTRACTOR_LABEL_CYCLE
    global DISTRACTOR_LABEL_CURSOR

    if not DISTRACTOR_LABEL_CYCLE:
        reset_balanced_distractor_label_sampler()

    label = DISTRACTOR_LABEL_CYCLE[DISTRACTOR_LABEL_CURSOR]
    DISTRACTOR_LABEL_CURSOR += 1

    if DISTRACTOR_LABEL_CURSOR >= len(DISTRACTOR_LABEL_CYCLE):
        random.shuffle(DISTRACTOR_LABEL_CYCLE)
        DISTRACTOR_LABEL_CURSOR = 0

    return label


def build_relation_distractor_sentences(
    object_pool: List[Dict[str, str]],
    used_uids: set,
    num_distractors: int,
) -> List[Dict[str, Any]]:
    """
    Build relation-based distractor sentences.

    Distractors may use all relation labels observed in the reference files,
    including:
        in, contains, on, supports, near

    They do not define the probe target and do not participate in the
    implicit support chain.

    Distractor logical relation labels are sampled from a balanced global
    cycle, so all reference labels are approximately equally frequent
    across the generated dataset.
    """
    if num_distractors <= 0:
        return []

    available_distractor_labels = get_available_distractor_relation_labels()

    if not available_distractor_labels:
        raise ValueError("No available distractor relation labels.")

    # Use objects outside the A/B/C support chain to avoid accidental
    # target leakage.
    candidate_pool = [
        obj for obj in object_pool
        if obj["uid"] not in used_uids
    ]

    if len(candidate_pool) < 2:
        raise ValueError(
            "Not enough non-chain objects to build relation distractors. "
            f"candidate_pool_size={len(candidate_pool)}"
        )

    distractors = []

    for _ in range(num_distractors):
        x, y = random.sample(candidate_pool, 2)

        # Balanced logical-label sampling.
        relation = sample_balanced_distractor_relation_label()

        sentence_info = realize_relation_sentence(
            subj=x["uid"],
            relation=relation,
            obj=y["uid"],
            force_surface_mode=None,
        )

        distractors.append({
            "sentence": sentence_info["sentence"],
            "subj_uid": x["uid"],
            "obj_uid": y["uid"],
            "logical_relation": relation,
            "sentence_role": "relation_distractor",

            "surface_mode": sentence_info["surface_mode"],
            "surface_relation": sentence_info["surface_relation"],
            "surface_subject_uid": sentence_info["surface_subject_uid"],
            "surface_object_uid": sentence_info["surface_object_uid"],
            "template": sentence_info["template"],
        })

    return distractors


def make_target_surface_forms(
    subj_uid: str,
    relation: str,
    obj_uid: str,
) -> List[str]:
    """
    Build possible explicit surface forms for the hidden target.
    Used only to avoid direct leakage into text.
    """
    ensure_templates_for_relation(relation)

    forms = []

    for template in DIRECT_SURFACE_TEMPLATES.get(relation, []):
        forms.append(template.format(subj=subj_uid, obj=obj_uid))

    for template in INVERSE_SURFACE_TEMPLATES.get(relation, []):
        forms.append(template.format(subj=subj_uid, obj=obj_uid))

    return sorted(set(forms))


def build_synthetic_sample(
    sample_index: int,
    scene: str,
    object_pool: List[Dict[str, str]],
    target_relation: str,
    implicit_rule: str,
) -> Dict[str, Any]:
    """
    Build one Step4T-style implicit-transitive synthetic sample.

    Target/support chain:
        only left_of/right_of/above/below

    Distractors:
        relation sentences sampled from all reference labels
    """

    if target_relation not in IMPLICIT_TARGET_LABELS:
        raise ValueError(
            f"Invalid target_relation={target_relation}. "
            f"Allowed target labels: {sorted(IMPLICIT_TARGET_LABELS)}"
        )

    A, B, C = choose_three_distinct_objects(object_pool)

    support_edges = build_support_edges(A, B, C, target_relation, implicit_rule)

    supporting_relations = []
    support_sentences = []

    for edge_idx, (subj, rel, obj) in enumerate(support_edges):
        if rel not in IMPLICIT_TARGET_LABELS:
            raise ValueError(
                f"Support relation {rel} is not a clean directional target label."
            )

        sentence_info = realize_relation_sentence(
            subj=subj["uid"],
            relation=rel,
            obj=obj["uid"],
            force_surface_mode=None,
        )

        supporting_relations.append(
            make_supporting_relation_record(
                subj=subj,
                relation=rel,
                obj=obj,
                sentence_info=sentence_info,
                template_index=edge_idx,
            )
        )

        support_sentences.append(sentence_info["sentence"])

    # Add distractor sentences so the synthetic paragraph length follows
    # the real Step4T reference sentence-count distribution.
    if REFERENCE_SENTENCE_COUNTS:
        target_sentence_count = random.choice(REFERENCE_SENTENCE_COUNTS)
    else:
        target_sentence_count = random.choice([14, 15, 16])

    num_distractors = max(0, target_sentence_count - len(support_sentences))

    used_uids = {
        A["uid"],
        B["uid"],
        C["uid"],
    }

    distractor_infos = build_relation_distractor_sentences(
        object_pool=object_pool,
        used_uids=used_uids,
        num_distractors=num_distractors,
    )

    distractor_sentences = [
        item["sentence"]
        for item in distractor_infos
    ]

    intro = random.choice(INTRO_TEMPLATES)

    all_sentences = support_sentences + distractor_sentences
    random.shuffle(all_sentences)

    text = intro + " " + " ".join(all_sentences)

    # Avoid directly writing the hidden target relation in the paragraph.
    for forbidden in make_target_surface_forms(A["uid"], target_relation, C["uid"]):
        if forbidden in text:
            raise ValueError(f"Target relation leaked into text: {forbidden}")

    subject_spans = find_all_char_spans(text, A["uid"])
    object_spans = find_all_char_spans(text, C["uid"])

    if not subject_spans or not object_spans:
        raise ValueError(
            f"Failed to locate probe subject/object spans. "
            f"A={A['uid']}, C={C['uid']}, text={text}"
        )

    sample_id = f"synthetic_step4T_{sample_index:06d}_{uuid.uuid4().hex[:8]}"

    paragraph_id = f"synthetic_{scene}_paragraph_{sample_index:06d}"
    chunk_id = f"synthetic_{scene}_chunk_{sample_index // 1000}"
    cluster_id = f"{chunk_id}_cluster_{sample_index % 10}"

    pair_group_id = (
        f"{paragraph_id}|{A['uid']}|{C['uid']}|"
        f"{target_relation}|implicit_transitive|{implicit_rule}"
    )

    row = {
        "sample_id": sample_id,
        "pair_group_id": pair_group_id,

        "scene": scene,
        "experiment": OUTPUT_BASENAME,
        "generation_mode": "synthetic",
        "paragraph_id": paragraph_id,
        "chunk_id": chunk_id,
        "cluster_id": cluster_id,
        "grid_i": None,
        "grid_j": None,
        "paragraph_index_within_cluster": sample_index,

        "text": text,

        "evidence": {
            "evidence_type": "implicit_transitive",
            "probe_direction_relative_to_text": "implicit",
            "has_explicit_evidence_sentence": False,
            "evidence_sentence": None,
            "sentence_index_in_paragraph": None,
            "surface_order": None,

            "supporting_relation_source": "synthetic_explicit_relations",
            "implicit_rule": implicit_rule,
            "anchor_uid": B["uid"],
            "supporting_relations": supporting_relations,
            "num_supporting_paths": 1,
            "original_pair_evidence_type": "synthetic_implicit_labeled",
            "label_source": "synthetic_controlled_transitive_generation",

            "distractor_source": "relation_templates_from_reference_labels",
            "distractor_relation_labels": [
                item["logical_relation"] for item in distractor_infos
            ],
            "distractor_surface_relations": [
                item["surface_relation"] for item in distractor_infos
            ],
            "distractor_sentences": [
                item["sentence"] for item in distractor_infos
            ],
            "distractor_sentence_infos": distractor_infos,

            "all_supporting_paths": [
                {
                    "implicit_rule": implicit_rule,
                    "anchor_uid": B["uid"],
                    "supporting_relations": supporting_relations,
                }
            ],
        },

        "probe_pair": {
            "probe_subject_uid": A["uid"],
            "probe_subject_id": A["id"],
            "probe_subject_type": A["type"],
            "probe_subject_mention_text": A["uid"],
            "probe_subject_all_char_spans_in_paragraph": subject_spans,
            "probe_subject_span_in_evidence_sentence": None,
            "probe_subject_span_in_paragraph": subject_spans[0],

            "probe_object_uid": C["uid"],
            "probe_object_id": C["id"],
            "probe_object_type": C["type"],
            "probe_object_mention_text": C["uid"],
            "probe_object_all_char_spans_in_paragraph": object_spans,
            "probe_object_span_in_evidence_sentence": None,
            "probe_object_span_in_paragraph": object_spans[0],

            "probe_relation_label": target_relation,
            "label_space": "clean_directional_spatial_relation",
            "has_relation_label": True,
            "is_explicit_in_text": False,
            "is_inverse_of_text_relation": False,
            "is_symmetric_relation": False,
            "is_directional_relation": True,
            "is_implicit_transitive": True,
        },

        "source_relation": {
            "relation_source": "synthetic_transitive_rule",
            "source_relation_summary": {
                "subject_id": A["id"],
                "relation": target_relation,
                "object_id": C["id"],
                "relation_family": "synthetic_geometric",
            },
            "label_source": "synthetic_controlled_transitive_generation",
            "sample_source": "synthetic_step4T_generator",
            "text_source": "synthetic_paragraph_text",
            "original_pair_target": {
                "subject_uid": A["uid"],
                "subject_id": A["id"],
                "subject_type": A["type"],
                "relation_label": target_relation,
                "object_uid": C["uid"],
                "object_id": C["id"],
                "object_type": C["type"],
                "pair_evidence_type": "synthetic_implicit_labeled",
                "relation_source": "synthetic_direct",
            },
        },

        "geometry": {
            "has_geometry_label": False,
            "geometry_label": None,
        },

        "source_step3_file": None,
        "source_step3_scene": scene,

        "synthetic_metadata": {
            "is_synthetic": True,
            "target_num_samples": TARGET_NUM_SAMPLES,
            "object_pool": A["pool"],
            "target_relation": target_relation,
            "implicit_rule": implicit_rule,
            "template_control": {
                "implicit_target_labels_restricted_to_clean_directional": True,
                "allowed_implicit_target_labels": sorted(IMPLICIT_TARGET_LABELS),
                "supporting_relations_restricted_to_target_labels": True,
                "distractors_use_all_reference_relation_labels": True,
                "distractor_relation_labels": sorted(DISTRACTOR_RELATION_LABELS),
                "uses_direct_and_inverse_surface_forms": True,
                "target_relation_not_explicitly_written": True,
                "train_test_object_names_disjoint": True,
            },
        },
    }

    return row


print("Synthetic sample generator loaded.")

Synthetic sample generator loaded.


In [7]:
# ============================================================
# Generate synthetic Step4T-style samples
#
# Target labels:
#   left_of, right_of, above, below
#
# Distractors:
#   relation sentences sampled from all reference labels
# ============================================================

def compute_label_targets(
    target_num_samples: int,
    labels: List[str],
    min_label_counts: Counter,
) -> Dict[str, int]:
    """
    Balanced target count construction.

    In the current clean directional setting:

        target labels = left_of, right_of, above, below

    TARGET_NUM_SAMPLES is the total number of implicit-target samples.
    With TARGET_NUM_SAMPLES=10000 and 4 target labels, each label receives:

        10000 / 4 = 2500 samples

    min_label_counts is intentionally ignored here because we want strict
    control over the new clean directional dataset size.
    """
    labels = list(labels)

    if not labels:
        raise ValueError("No labels available for schedule construction.")

    disallowed_labels = sorted(set(labels) - IMPLICIT_TARGET_LABELS)
    if disallowed_labels:
        raise ValueError(
            "Non-directional labels were passed into implicit target generation: "
            f"{disallowed_labels}"
        )

    base = int(target_num_samples) // len(labels)
    remainder = int(target_num_samples) % len(labels)

    targets = {}
    for i, label in enumerate(labels):
        targets[label] = base + (1 if i < remainder else 0)

    return targets


def expand_balanced_items(
    n: int,
    scenes: List[str],
    label: str,
    object_pool: List[Dict[str, str]],
    split_name: str,
) -> List[Dict[str, Any]]:
    """
    Spread a fixed number of samples for one label across scenes and
    valid implicit rules as evenly as possible.
    """
    rules = valid_implicit_rules_for_label(label)

    if not rules:
        raise ValueError(
            f"No valid implicit rules for label={label}. "
            f"Allowed target labels: {sorted(IMPLICIT_TARGET_LABELS)}"
        )

    combos = [
        {
            "split": split_name,
            "scene": scene,
            "label": label,
            "implicit_rule": rule,
            "pool": object_pool,
        }
        for scene in scenes
        for rule in rules
    ]

    random.shuffle(combos)

    items = []
    for i in range(n):
        items.append(combos[i % len(combos)])

    return items


# Final target-label safety check.
LABELS = [
    "left_of",
    "right_of",
    "above",
    "below",
]

if set(LABELS) != IMPLICIT_TARGET_LABELS:
    raise ValueError(
        "Final target LABELS do not match the required clean directional set.\n"
        f"Expected: {sorted(IMPLICIT_TARGET_LABELS)}\n"
        f"Actual:   {sorted(LABELS)}"
    )

label_targets = compute_label_targets(
    target_num_samples=TARGET_NUM_SAMPLES,
    labels=LABELS,
    min_label_counts=current_synthetic_label_counts,
)

EFFECTIVE_TARGET_NUM_SAMPLES = sum(label_targets.values())

print("Requested minimum TARGET_NUM_SAMPLES:", TARGET_NUM_SAMPLES)
print("Effective target samples:", EFFECTIVE_TARGET_NUM_SAMPLES)

print("\nImplicit target labels:")
print(LABELS)

print("\nDistractor relation labels:")
print(DISTRACTOR_RELATION_LABELS)

print("\nAll reference label counts:")
print(dict(reference_label_counts))

print("\nTarget reference label counts:")
print({label: reference_label_counts.get(label, 0) for label in LABELS})

print("\nOld synthetic lower-bound counts:")
print(dict(current_synthetic_label_counts))

print("\nTarget old synthetic lower-bound counts:")
print({label: current_synthetic_label_counts.get(label, 0) for label in LABELS})

print("\nBalanced target label counts:")
print(label_targets)

schedule = []

for label, label_count in label_targets.items():
    n_train = int(round(label_count * 0.8))
    n_test = label_count - n_train

    schedule.extend(
        expand_balanced_items(
            n=n_train,
            scenes=TRAIN_SCENES,
            label=label,
            object_pool=TRAIN_OBJECTS,
            split_name="train",
        )
    )

    schedule.extend(
        expand_balanced_items(
            n=n_test,
            scenes=TEST_SCENES,
            label=label,
            object_pool=TEST_OBJECTS,
            split_name="test",
        )
    )

random.shuffle(schedule)

reset_balanced_distractor_label_sampler()

synthetic_rows = []

for i, item in enumerate(schedule):
    row = build_synthetic_sample(
        sample_index=i,
        scene=item["scene"],
        object_pool=item["pool"],
        target_relation=item["label"],
        implicit_rule=item["implicit_rule"],
    )
    synthetic_rows.append(row)

print("Generated synthetic rows:", len(synthetic_rows))

label_counts = Counter(r["probe_pair"]["probe_relation_label"] for r in synthetic_rows)

support_relation_counts = Counter()
distractor_relation_counts = Counter()
distractor_surface_relation_counts = Counter()

for r in synthetic_rows:
    for sr in r["evidence"]["supporting_relations"]:
        support_relation_counts[sr["relation"]] += 1

    for rel in r["evidence"].get("distractor_relation_labels", []):
        distractor_relation_counts[rel] += 1

    for rel in r["evidence"].get("distractor_surface_relations", []):
        distractor_surface_relation_counts[rel] += 1

print("\nTarget label counts:")
print(label_counts)

print("\nSupport relation counts:")
print(support_relation_counts)

print("\nDistractor logical relation counts:")
print(distractor_relation_counts)

if distractor_relation_counts:
    distractor_min = min(distractor_relation_counts.values())
    distractor_max = max(distractor_relation_counts.values())

    print("\nDistractor logical-label balance check:")
    print("min distractor logical-label count:", distractor_min)
    print("max distractor logical-label count:", distractor_max)
    print("max - min:", distractor_max - distractor_min)
    print("num distractor logical labels:", len(distractor_relation_counts))
    print("expected distractor labels:", sorted(DISTRACTOR_RELATION_LABELS))
    print(
        "missing distractor labels:",
        sorted(set(DISTRACTOR_RELATION_LABELS) - set(distractor_relation_counts.keys())),
    )

print("\nDistractor surface relation counts:")
print(distractor_surface_relation_counts)

print("\nImplicit rule counts:")
print(Counter(r["evidence"]["implicit_rule"] for r in synthetic_rows))

print("\nScene counts:")
print(Counter(r["scene"] for r in synthetic_rows))

print("\nScene x target label counts:")
print(Counter((r["scene"], r["probe_pair"]["probe_relation_label"]) for r in synthetic_rows))

print("\nScene x implicit_rule counts:")
print(Counter((r["scene"], r["evidence"]["implicit_rule"]) for r in synthetic_rows))

print("\nObject pool counts:")
print(Counter(r["synthetic_metadata"]["object_pool"] for r in synthetic_rows))

print("\nSynthetic text sentence-count distribution:")
print(Counter(count_sentence_like_units(r["text"]) for r in synthetic_rows))

synthetic_char_lengths = [len(r["text"]) for r in synthetic_rows]
print("\nSynthetic text char length min/mean/max:")
print(
    min(synthetic_char_lengths),
    round(sum(synthetic_char_lengths) / len(synthetic_char_lengths), 2),
    max(synthetic_char_lengths),
)

print("\nTarget label balance check:")
print("min target label count:", min(label_counts.values()))
print("max target label count:", max(label_counts.values()))
print("max - min:", max(label_counts.values()) - min(label_counts.values()))

print("\nOld synthetic lower-bound check for target labels only:")
for label in LABELS:
    old_count = current_synthetic_label_counts.get(label, 0)
    print(
        f"{label}: new={label_counts[label]}, "
        f"old={old_count}, "
        f"ok={label_counts[label] >= old_count}"
    )

Requested minimum TARGET_NUM_SAMPLES: 10000
Effective target samples: 10000

Implicit target labels:
['left_of', 'right_of', 'above', 'below']

Distractor relation labels:
['above', 'below', 'contains', 'in', 'left_of', 'near', 'on', 'right_of', 'supports']

All reference label counts:
{'above': 330, 'right_of': 951, 'contains': 48, 'on': 435, 'supports': 435, 'below': 330, 'in': 48, 'left_of': 951, 'near': 254}

Target reference label counts:
{'left_of': 951, 'right_of': 951, 'above': 330, 'below': 330}

Old synthetic lower-bound counts:
{'right_of': 2501, 'left_of': 2499}

Target old synthetic lower-bound counts:
{'left_of': 2499, 'right_of': 2501, 'above': 0, 'below': 0}

Balanced target label counts:
{'left_of': 2500, 'right_of': 2500, 'above': 2500, 'below': 2500}
Balanced distractor label sampler initialized.
Distractor label cycle: ['on', 'left_of', 'supports', 'below', 'right_of', 'near', 'in', 'contains', 'above']
Generated synthetic rows: 10000

Target label counts:
Counter({

In [9]:
# ============================================================
# Quality checks
# ============================================================

def check_target_not_explicit(row: Dict[str, Any]) -> bool:
    text = row["text"]
    pp = row["probe_pair"]

    A = pp["probe_subject_uid"]
    C = pp["probe_object_uid"]
    relation = pp["probe_relation_label"]

    forbidden_patterns = make_target_surface_forms(A, relation, C)

    return not any(p in text for p in forbidden_patterns)


def check_spans(row: Dict[str, Any]) -> bool:
    text = row["text"]
    pp = row["probe_pair"]

    for key, mention_key in [
        ("probe_subject_span_in_paragraph", "probe_subject_mention_text"),
        ("probe_object_span_in_paragraph", "probe_object_mention_text"),
    ]:
        span = pp[key]
        mention = pp[mention_key]

        if span is None:
            return False

        start, end = span["start"], span["end"]

        if start < 0 or end <= start or end > len(text):
            return False

        if text[start:end] != mention:
            return False

    return True


def check_label_coverage_and_balance(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Check target-label coverage and balance.

    In the current clean directional setting, old synthetic lower-bound
    counts are not used as constraints. Target counts are fixed by
    TARGET_NUM_SAMPLES_PER_TARGET_LABEL.
    """
    counts = Counter(r["probe_pair"]["probe_relation_label"] for r in rows)

    missing_labels = sorted(set(LABELS) - set(counts.keys()))
    extra_labels = sorted(set(counts.keys()) - set(LABELS))

    min_count = min(counts.values()) if counts else 0
    max_count = max(counts.values()) if counts else 0

    expected_per_label = TARGET_NUM_SAMPLES_PER_TARGET_LABEL

    count_deviations = {
        label: {
            "expected": expected_per_label,
            "observed": counts.get(label, 0),
            "difference": counts.get(label, 0) - expected_per_label,
        }
        for label in LABELS
    }

    return {
        "counts": counts,
        "missing_labels": missing_labels,
        "extra_labels": extra_labels,
        "min_count": min_count,
        "max_count": max_count,
        "max_minus_min": max_count - min_count,
        "expected_per_label": expected_per_label,
        "count_deviations": count_deviations,
    }


# ------------------------------------------------------------
# Basic leakage/span checks
# ------------------------------------------------------------

leaked = [r["sample_id"] for r in synthetic_rows if not check_target_not_explicit(r)]
bad_spans = [r["sample_id"] for r in synthetic_rows if not check_spans(r)]

print("Target leakage count:", len(leaked))
print("Bad span count:", len(bad_spans))

if leaked[:5]:
    print("Example leaked ids:", leaked[:5])

if bad_spans[:5]:
    print("Example bad span ids:", bad_spans[:5])

if leaked:
    raise ValueError(f"Found {len(leaked)} samples with possible target leakage.")

if bad_spans:
    raise ValueError(f"Found {len(bad_spans)} samples with bad spans.")


# ------------------------------------------------------------
# Target label coverage and balance
# ------------------------------------------------------------

label_check = check_label_coverage_and_balance(synthetic_rows)

print("\nTarget label coverage and balance:")
print("Target label counts:", label_check["counts"])
print("Missing target labels:", label_check["missing_labels"])
print("Extra target labels:", label_check["extra_labels"])
print("Expected per target label:", label_check["expected_per_label"])
print("Count deviations:", label_check["count_deviations"])
print("Min target label count:", label_check["min_count"])
print("Max target label count:", label_check["max_count"])
print("Max - min:", label_check["max_minus_min"])

if label_check["missing_labels"]:
    raise ValueError(f"Missing target labels: {label_check['missing_labels']}")

if label_check["extra_labels"]:
    raise ValueError(f"Unexpected target labels: {label_check['extra_labels']}")

if label_check["max_minus_min"] > 1:
    raise ValueError(
        f"Target labels are not balanced enough: {label_check['counts']}"
    )


# ------------------------------------------------------------
# Check that target and support relations are clean directional only
# ------------------------------------------------------------

target_labels_observed = Counter()
support_labels_observed = Counter()
distractor_labels_observed = Counter()
distractor_surface_labels_observed = Counter()

for row in synthetic_rows:
    target_labels_observed[row["probe_pair"]["probe_relation_label"]] += 1

    for sr in row["evidence"]["supporting_relations"]:
        support_labels_observed[sr["relation"]] += 1

    for rel in row["evidence"].get("distractor_relation_labels", []):
        distractor_labels_observed[rel] += 1

    for rel in row["evidence"].get("distractor_surface_relations", []):
        distractor_surface_labels_observed[rel] += 1

bad_target_labels = sorted(set(target_labels_observed.keys()) - IMPLICIT_TARGET_LABELS)
bad_support_labels = sorted(set(support_labels_observed.keys()) - IMPLICIT_TARGET_LABELS)

print("\nObserved target labels:")
print(target_labels_observed)

print("\nObserved support labels:")
print(support_labels_observed)

print("\nObserved distractor logical labels:")
print(distractor_labels_observed)

print("\nObserved distractor surface labels:")
print(distractor_surface_labels_observed)

print("\nBad target labels:", bad_target_labels)
print("Bad support labels:", bad_support_labels)

if bad_target_labels:
    raise ValueError(f"Non-directional labels found as targets: {bad_target_labels}")

if bad_support_labels:
    raise ValueError(f"Non-directional labels found in supporting chains: {bad_support_labels}")


# ------------------------------------------------------------
# Check distractor logical-label coverage and balance
# ------------------------------------------------------------

missing_distractor_reference_labels = sorted(
    set(DISTRACTOR_RELATION_LABELS) - set(distractor_labels_observed.keys())
)

extra_distractor_labels = sorted(
    set(distractor_labels_observed.keys()) - set(DISTRACTOR_RELATION_LABELS)
)

if distractor_labels_observed:
    distractor_min = min(distractor_labels_observed.values())
    distractor_max = max(distractor_labels_observed.values())
else:
    distractor_min = 0
    distractor_max = 0

print("\nReference labels allowed as distractors:")
print(DISTRACTOR_RELATION_LABELS)

print("\nReference labels not observed in distractors:")
print(missing_distractor_reference_labels)

print("\nUnexpected distractor logical labels:")
print(extra_distractor_labels)

print("\nDistractor logical-label balance:")
print("min:", distractor_min)
print("max:", distractor_max)
print("max - min:", distractor_max - distractor_min)

if missing_distractor_reference_labels:
    raise ValueError(
        f"Some reference labels were never used as distractor logical labels: "
        f"{missing_distractor_reference_labels}"
    )

if extra_distractor_labels:
    raise ValueError(
        f"Unexpected distractor logical labels found: {extra_distractor_labels}"
    )

if distractor_max - distractor_min > 1:
    raise ValueError(
        "Distractor logical labels are not balanced enough. "
        f"Counts: {distractor_labels_observed}"
    )


# ------------------------------------------------------------
# Check train/test object overlap by scene
# ------------------------------------------------------------

train_used = set()
test_used = set()

for row in synthetic_rows:
    pool_name = row["synthetic_metadata"]["object_pool"]

    pp = row["probe_pair"]
    uids = {
        pp["probe_subject_uid"],
        pp["probe_object_uid"],
    }

    for sr in row["evidence"]["supporting_relations"]:
        uids.add(sr["subject_uid"])
        uids.add(sr["object_uid"])

    if pool_name == TRAIN_POOL_NAME:
        train_used.update(uids)
    elif pool_name == TEST_POOL_NAME:
        test_used.update(uids)

overlap = train_used & test_used

print("\nTrain used objects:", len(train_used))
print("Test used objects:", len(test_used))
print("Train/test object overlap:", len(overlap))

if overlap:
    raise ValueError(f"Train/test object overlap detected: {list(sorted(overlap))[:10]}")


# ------------------------------------------------------------
# Surface form diagnostic for supporting relations
# ------------------------------------------------------------

surface_mode_counts = Counter()
label_surface_counts = Counter()
template_counts = Counter()

for row in synthetic_rows:
    label = row["probe_pair"]["probe_relation_label"]

    for sr in row["evidence"]["supporting_relations"]:
        surface_mode_counts[sr["surface_mode"]] += 1
        label_surface_counts[(label, sr["surface_mode"])] += 1
        template_counts[sr["template"]] += 1

print("\nSupport surface mode counts:")
print(surface_mode_counts)

print("\nTarget label x support surface mode counts:")
print(label_surface_counts)

print("\nNumber of unique supporting templates:", len(template_counts))
print("Top supporting templates:")
for tpl, count in template_counts.most_common(10):
    print(count, " :: ", tpl)

print("\nQuality checks passed.")

Target leakage count: 0
Bad span count: 0

Target label coverage and balance:
Target label counts: Counter({'below': 2500, 'right_of': 2500, 'left_of': 2500, 'above': 2500})
Missing target labels: []
Extra target labels: []
Expected per target label: 2500
Count deviations: {'left_of': {'expected': 2500, 'observed': 2500, 'difference': 0}, 'right_of': {'expected': 2500, 'observed': 2500, 'difference': 0}, 'above': {'expected': 2500, 'observed': 2500, 'difference': 0}, 'below': {'expected': 2500, 'observed': 2500, 'difference': 0}}
Min target label count: 2500
Max target label count: 2500
Max - min: 0

Observed target labels:
Counter({'below': 2500, 'right_of': 2500, 'left_of': 2500, 'above': 2500})

Observed support labels:
Counter({'left_of': 5003, 'above': 5001, 'below': 4999, 'right_of': 4997})

Observed distractor logical labels:
Counter({'supports': 13734, 'below': 13734, 'on': 13733, 'left_of': 13733, 'right_of': 13733, 'near': 13733, 'in': 13733, 'contains': 13733, 'above': 13733

In [10]:
# ============================================================
# Save synthetic dataset
# ============================================================

output_json_path = OUTPUT_DIR / OUTPUT_JSON
output_step5_compat_json_path = OUTPUT_DIR / STEP5_COMPAT_JSON
output_summary_path = OUTPUT_DIR / f"{OUTPUT_BASENAME}_summary.json"
output_zip_path = Path("/content") / OUTPUT_ZIP

# Main requested filename.
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(synthetic_rows, f, indent=2, ensure_ascii=False)

# Step5-compatible filename.
with open(output_step5_compat_json_path, "w", encoding="utf-8") as f:
    json.dump(synthetic_rows, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------
# Final target-label statistics
# ------------------------------------------------------------

final_label_counts = Counter(
    r["probe_pair"]["probe_relation_label"]
    for r in synthetic_rows
)

final_min_label_count = min(final_label_counts.values()) if final_label_counts else 0
final_max_label_count = max(final_label_counts.values()) if final_label_counts else 0

# ------------------------------------------------------------
# Final support and distractor statistics
# ------------------------------------------------------------

final_support_relation_counts = Counter()
final_distractor_relation_counts = Counter()
final_distractor_surface_relation_counts = Counter()

for row in synthetic_rows:
    for sr in row["evidence"]["supporting_relations"]:
        final_support_relation_counts[sr["relation"]] += 1

    for rel in row["evidence"].get("distractor_relation_labels", []):
        final_distractor_relation_counts[rel] += 1

    for rel in row["evidence"].get("distractor_surface_relations", []):
        final_distractor_surface_relation_counts[rel] += 1

# ------------------------------------------------------------
# Old synthetic count comparison
#
# This is diagnostic only.
# Old synthetic counts are not used as lower bounds in the clean
# directional setting because target counts are fixed by
# TARGET_NUM_SAMPLES_PER_TARGET_LABEL.
# ------------------------------------------------------------

old_synthetic_count_comparison = {
    label: {
        "old": old_count,
        "new": final_label_counts.get(label, 0),
    }
    for label, old_count in current_synthetic_label_counts.items()
    if label in LABELS
}

# ------------------------------------------------------------
# Distractor logical-label balance statistics
#
# This checks evidence["distractor_relation_labels"], not
# evidence["distractor_surface_relations"].
#
# Surface labels may be less balanced because inverse surface forms can
# swap labels such as:
#   left_of -> right_of
#   in      -> contains
# ------------------------------------------------------------

final_distractor_min_label_count = (
    min(final_distractor_relation_counts.values())
    if final_distractor_relation_counts
    else 0
)

final_distractor_max_label_count = (
    max(final_distractor_relation_counts.values())
    if final_distractor_relation_counts
    else 0
)

missing_distractor_labels = sorted(
    set(DISTRACTOR_RELATION_LABELS) - set(final_distractor_relation_counts.keys())
)

# ------------------------------------------------------------
# Summary payload
# ------------------------------------------------------------

summary = {
    "output_file": OUTPUT_JSON,
    "step5_compatible_output_file": STEP5_COMPAT_JSON,
    "num_samples": len(synthetic_rows),
    "random_seed": RANDOM_SEED,
    "target_num_samples": TARGET_NUM_SAMPLES,
    "target_num_samples_per_label": TARGET_NUM_SAMPLES_PER_TARGET_LABEL,
    "effective_target_num_samples": EFFECTIVE_TARGET_NUM_SAMPLES,
    "format": "Step4T-style implicit_transitive probe samples",

    "experiment_setting": {
        "clean_directional_implicit_targets": True,
        "implicit_target_labels": LABELS,
        "implicit_target_label_set": sorted(IMPLICIT_TARGET_LABELS),
        "target_samples_per_label": TARGET_NUM_SAMPLES_PER_TARGET_LABEL,
        "supporting_chain_labels_restricted_to_target_labels": True,

        "distractors_use_all_reference_relation_labels": True,
        "balanced_distractor_logical_label_sampling": True,
        "distractor_relation_labels": sorted(DISTRACTOR_RELATION_LABELS),
        "all_reference_labels": sorted(ALL_REFERENCE_LABELS),
        "excluded_from_implicit_target_but_allowed_as_distractor": sorted(
            set(DISTRACTOR_RELATION_LABELS) - set(LABELS)
        ),
    },

    "reference_label_counts": dict(reference_label_counts),
    "current_synthetic_reference_label_counts": dict(current_synthetic_label_counts),
    "old_synthetic_count_comparison": old_synthetic_count_comparison,
    "label_targets": dict(label_targets),

    "label_counts": dict(final_label_counts),
    "support_relation_counts": dict(final_support_relation_counts),
    "distractor_relation_counts": dict(final_distractor_relation_counts),
    "distractor_surface_relation_counts": dict(final_distractor_surface_relation_counts),

    "label_balance": {
        "min_label_count": final_min_label_count,
        "max_label_count": final_max_label_count,
        "max_minus_min": final_max_label_count - final_min_label_count,
    },

    "distractor_relation_balance": {
        "min_label_count": final_distractor_min_label_count,
        "max_label_count": final_distractor_max_label_count,
        "max_minus_min": final_distractor_max_label_count - final_distractor_min_label_count,
        "num_observed_labels": len(final_distractor_relation_counts),
        "expected_labels": sorted(DISTRACTOR_RELATION_LABELS),
        "missing_labels": missing_distractor_labels,
    },

    "implicit_rule_counts": dict(
        Counter(r["evidence"]["implicit_rule"] for r in synthetic_rows)
    ),

    "scene_counts": dict(
        Counter(r["scene"] for r in synthetic_rows)
    ),

    "scene_label_counts": {
        f"{scene}|{label}": count
        for (scene, label), count in Counter(
            (r["scene"], r["probe_pair"]["probe_relation_label"])
            for r in synthetic_rows
        ).items()
    },

    "scene_implicit_rule_counts": {
        f"{scene}|{rule}": count
        for (scene, rule), count in Counter(
            (r["scene"], r["evidence"]["implicit_rule"])
            for r in synthetic_rows
        ).items()
    },

    "object_pool_counts": dict(
        Counter(r["synthetic_metadata"]["object_pool"] for r in synthetic_rows)
    ),

    "train_scenes": TRAIN_SCENES,
    "test_scenes": TEST_SCENES,
    "train_used_object_count": len(train_used),
    "test_used_object_count": len(test_used),
    "train_test_object_overlap": len(overlap),

    "surface_mode_counts": dict(surface_mode_counts),

    "label_surface_counts": {
        f"{label}|{mode}": count
        for (label, mode), count in label_surface_counts.items()
    },

    "num_unique_supporting_templates": len(template_counts),

    "design_controls": {
        "target_relation_labels_are_clean_directional_only": (
            set(final_label_counts.keys()) == set(LABELS) == IMPLICIT_TARGET_LABELS
        ),

        "supporting_relation_labels_are_clean_directional_only": (
            set(final_support_relation_counts.keys()) <= IMPLICIT_TARGET_LABELS
        ),

        "distractors_use_reference_relation_labels": True,
        "non_directional_reference_labels_allowed_only_as_distractors": True,

        "target_label_distribution_balanced": (
            final_max_label_count - final_min_label_count
        ) <= 1,

        "target_num_samples_fixed_by_per_label_count": True,

        "distractor_logical_relation_distribution_balanced_by_cycle": (
            len(missing_distractor_labels) == 0
            and (
                final_distractor_max_label_count
                - final_distractor_min_label_count
            ) <= 1
        ),

        "text_length_matched_to_real_reference_sentence_counts": True,
        "target_relation_not_explicitly_written": True,
        "implicit_rule_balanced_within_valid_rules": True,
        "scene_label_balanced_by_schedule": True,
        "direct_inverse_surface_forms_mixed": True,
        "train_test_object_names_disjoint": True,
        "main_output_filename_preserved": True,
        "step5_compatible_copy_saved": True,
    },
}

# ------------------------------------------------------------
# Save summary JSON
# ------------------------------------------------------------

with open(output_summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------
# Zip output files
# ------------------------------------------------------------

if output_zip_path.exists():
    output_zip_path.unlink()

with zipfile.ZipFile(str(output_zip_path), "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(str(output_json_path), arcname=OUTPUT_JSON)
    zf.write(str(output_step5_compat_json_path), arcname=STEP5_COMPAT_JSON)
    zf.write(str(output_summary_path), arcname=f"{OUTPUT_BASENAME}_summary.json")

print("Saved synthetic dataset:")
print(output_json_path)

print("\nSaved Step5-compatible copy:")
print(output_step5_compat_json_path)

print("\nSaved summary:")
print(output_summary_path)

print("\nCreated zip:")
print(output_zip_path)

print("\nSummary:")
print(json.dumps(summary, indent=2, ensure_ascii=False))

Saved synthetic dataset:
/content/pilot4_synthetic_disorder_step4_lrab_25000/output/pilot4_synthetic_disorder_step4_lrab_25000.json

Saved Step5-compatible copy:
/content/pilot4_synthetic_disorder_step4_lrab_25000/output/step4T_pilot4_synthetic_disorder_step4_lrab_25000_implicit_transitive_probe_samples.json

Saved summary:
/content/pilot4_synthetic_disorder_step4_lrab_25000/output/pilot4_synthetic_disorder_step4_lrab_25000_summary.json

Created zip:
/content/pilot4_synthetic_disorder_step4_lrab_25000.zip

Summary:
{
  "output_file": "pilot4_synthetic_disorder_step4_lrab_25000.json",
  "step5_compatible_output_file": "step4T_pilot4_synthetic_disorder_step4_lrab_25000_implicit_transitive_probe_samples.json",
  "num_samples": 10000,
  "random_seed": 42,
  "target_num_samples": 10000,
  "target_num_samples_per_label": 2500,
  "effective_target_num_samples": 10000,
  "format": "Step4T-style implicit_transitive probe samples",
  "experiment_setting": {
    "clean_directional_implicit_target

In [11]:
# ============================================================
# Download synthetic dataset zip
# ============================================================

files.download(str(output_zip_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>